# Cryoablation dosimetry: thermal model and dose-response

Reproduces the quantities the manuscript reports: the finite-difference
thermal model checked against measured thermocouples, the hierarchical
dose-response fits, the dose metrics derived from them, and the lung-adapted
ice-ball geometry.

Inputs are in `data/` (per-bin cell counts with their assigned minimum
temperatures) and in the thermocouple directories. Sampling takes a few
minutes per model.

In [1]:
import numpy as np
import pandas as pd

import cryo_thermal as ct
import dose_response as dr

pd.set_option('display.width', 120)
print('NUTS backend:', dr.NUTS_BACKEND or 'pymc default')

NUTS backend: nutpie


## 1. Thermal model against measured thermocouples

One offset (dT, the gap between the boundary thermocouple and the true probe
surface) is fitted per experiment; every other property is fixed at the
literature value. R2 and mean absolute error are reported at the validation
thermocouples, which are not used in the fit.

In [2]:
EXPS = {e['id']: e for e in ct.build_experiments('.')}

for exp_id in ('celsio_A549_N1_n1', 'ice2x6_N1_c1', 'clinical_N1_c3'):
    res = ct.run_experiment(EXPS[exp_id])
    print(f"{exp_id:22s} dT = {res['dT']:7.2f} C   "
          f"R2_val = {res['R2_val']:6.3f}   MAE_val = {res['MAE_val']:5.2f} C")

celsio_A549_N1_n1      dT =  -60.44 C   R2_val =  0.719   MAE_val =  4.24 C


ice2x6_N1_c1           dT =    4.84 C   R2_val =  0.936   MAE_val =  3.69 C


clinical_N1_c3         dT =  -44.84 C   R2_val =  0.804   MAE_val = 11.13 C


## 2. Dose-response

Per radial bin, `n_expected = max(n_baseline, n_post_total)` and
`death = (n_expected - n_live) / n_expected`, so `n_live` is a count out of
`n_expected`. The observation model is beta-binomial on those counts, with a
three-parameter logistic mean and Gaussian replicate effects.

In [3]:
celsio = pd.read_csv('data/celsio_bins.csv')
ice = pd.read_csv('data/icesphere_bins.csv')
print(f'Celsio rows: {len(celsio)}   IceSphere rows: {len(ice)}')

Celsio rows: 810   IceSphere rows: 360


In [4]:
# Cell lines, temperature domain.
rows_A, groups_A, rep2g_A = dr.build_rows(
    celsio.dropna(subset=['T_min']), x_col='T_min', group_col='cell_line',
    replicate_col='bio_replicate', bin_size=2.0)

# The spread of bin denominators is what makes a single Gaussian variance
# inappropriate: binomial noise on a death fraction scales with 1/sqrt(n).
n = rows_A['n_expected']
print(f'{len(rows_A)} bins, denominators {n.min()} to {n.max()} (median {n.median():.0f}), '
      f'{int((n < 50).sum())} below 50')
print('binomial SD of a death fraction at p = 0.5: '
      f'{100 * np.sqrt(0.25 / 10):.1f} % at n = 10, '
      f'{100 * np.sqrt(0.25 / 1000):.1f} % at n = 1000')

model_A = dr.build_model(rows_A, len(groups_A), rep2g_A, 'betabinomial',
                         domain='temperature')
idata_A = dr.sample(model_A)
print(groups_A)

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`


WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


359 bins, denominators 3 to 3717 (median 862), 6 below 50
binomial SD of a death fraction at p = 0.5: 15.8 % at n = 10, 1.6 % at n = 1000


['A549', 'Calu-1', 'Calu-6']


In [5]:
# Platforms and protocols, temperature domain. The triple-freeze curve is the
# one the clinical projection uses.
_celsio_a549 = (celsio[celsio.cell_line == 'A549']
                .assign(cond='Celsio A549', rep=lambda d: d['bio_replicate']))
_ice = (ice[ice.label.isin(['2x6 F1', 'Clinical'])]
        .assign(cond=lambda d: d['label'].map({'2x6 F1': 'IceSphere F1',
                                               'Clinical': 'IceSphere triple'}),
                rep=lambda d: d['replicate']))
combined = pd.concat([_celsio_a549[['cond', 'rep', 'T_min', 'n_live', 'n_expected']],
                      _ice[['cond', 'rep', 'T_min', 'n_live', 'n_expected']]],
                     ignore_index=True)

rows_B, groups_B, rep2g_B = dr.build_rows(
    combined.dropna(subset=['T_min']), x_col='T_min', group_col='cond',
    replicate_col='rep', bin_size=2.0)

model_B = dr.build_model(rows_B, len(groups_B), rep2g_B, 'betabinomial',
                         domain='temperature')
idata_B = dr.sample(model_B)
print(groups_B)

['Celsio A549', 'IceSphere F1', 'IceSphere triple']


In [6]:
# Convergence.
for name, idata in (('cell lines', idata_A), ('platforms', idata_B)):
    d = dr.diagnostics(idata)
    print(f"{name:11s} max r_hat {d['r_hat'].max():.3f}   "
          f"min bulk ESS {d['ess_bulk'].min():.0f}   "
          f"min tail ESS {d['ess_tail'].min():.0f}   "
          f"max MCSE {d['mcse_mean'].max():.3f}")

cell lines  max r_hat 1.000   min bulk ESS 1707   min tail ESS 1846   max MCSE 0.124
platforms   max r_hat 1.000   min bulk ESS 1557   min tail ESS 2114   max MCSE 0.111


## 3. Dose metrics

The fitted plateau is below 100 % in every condition, so a threshold can be
read relative to that plateau or on an absolute percentage scale. Relative is
used throughout: ALD_q = x0 + ln(100/q - 1)/k, so ALD50 is the inflection
point by construction and ALD95 = x0 - 2.944/k. Absolute values are reported
alongside, with the fraction of posterior draws for which an absolute 95 %
threshold is not attainable.

In [7]:
def dose_table(idata, groups):
    out = []
    post = idata.posterior
    for i, g in enumerate(groups):
        L = post['L'].stack(s=('chain', 'draw')).values[i]
        k = post['k'].stack(s=('chain', 'draw')).values[i]
        rel50 = dr.posterior_dose(idata, i, 50, 'relative')
        rel95 = dr.posterior_dose(idata, i, 95, 'relative')
        abs95 = dr.posterior_dose(idata, i, 95, 'absolute')
        lo50, hi50 = dr.hdi_of(rel50)
        lo95, hi95 = dr.hdi_of(rel95)
        out.append(dict(
            group=g, L=L.mean(), k=k.mean(),
            ALD50=np.nanmean(rel50), ALD50_lo=lo50, ALD50_hi=hi50,
            ALD95=np.nanmean(rel95), ALD95_lo=lo95, ALD95_hi=hi95,
            ALD95_absolute=(np.nanmean(abs95) if np.isfinite(abs95).any()
                            else np.nan),
            pct_absolute_undefined=100.0 * np.mean(~np.isfinite(abs95))))
    return pd.DataFrame(out)


tab_A = dose_table(idata_A, groups_A)
tab_B = dose_table(idata_B, groups_B)
print('Cell lines (degC)')
print(tab_A.to_string(index=False, float_format=lambda v: f'{v:8.2f}'))
print('\nPlatforms and protocols (degC)')
print(tab_B.to_string(index=False, float_format=lambda v: f'{v:8.2f}'))

Cell lines (degC)
 group        L        k    ALD50  ALD50_lo  ALD50_hi    ALD95  ALD95_lo  ALD95_hi  ALD95_absolute  pct_absolute_undefined
  A549    89.07     0.12   -23.34    -30.11    -16.85   -47.17    -55.12    -39.93             NaN                  100.00
Calu-1    94.04     0.13    -6.38    -16.65      3.83   -29.69    -40.41    -18.78          -57.22                   89.18
Calu-6    95.93     0.13   -12.95    -20.81     -6.25   -35.40    -43.22    -27.62          -49.22                    5.98

Platforms and protocols (degC)
           group        L        k    ALD50  ALD50_lo  ALD50_hi    ALD95  ALD95_lo  ALD95_hi  ALD95_absolute  pct_absolute_undefined
     Celsio A549    89.39     0.13   -23.40    -30.27    -17.18   -45.63    -53.04    -38.45             NaN                  100.00
    IceSphere F1    95.10     0.09    -8.47    -13.95     -3.58   -41.19    -48.58    -34.35          -66.82                   44.78
IceSphere triple    96.58     0.20    -9.77    -18.37     -

In [8]:
# Directional contrasts.
def p_less(idata, groups, a, b, var='x0'):
    v = idata.posterior[var].stack(s=('chain', 'draw')).values
    return float(np.mean(v[groups.index(a)] < v[groups.index(b)]))


print('P(ALD50 A549 < Calu-1)      ', f"{p_less(idata_A, groups_A, 'A549', 'Calu-1'):.3f}")
print('P(ALD50 A549 < Calu-6)      ', f"{p_less(idata_A, groups_A, 'A549', 'Calu-6'):.3f}")
print('P(ALD50 Celsio < IceSphere) ',
      f"{p_less(idata_B, groups_B, 'Celsio A549', 'IceSphere F1'):.3f}")
print('P(plateau Celsio < IceSphere)',
      f"{p_less(idata_B, groups_B, 'Celsio A549', 'IceSphere F1', var='L'):.3f}")

P(ALD50 A549 < Calu-1)       0.994
P(ALD50 A549 < Calu-6)       0.981
P(ALD50 Celsio < IceSphere)  0.999
P(plateau Celsio < IceSphere) 1.000


## 4. Lung-adapted geometry

The solver is reparameterised with lung tissue properties and run through the
clinical three-cycle protocol. The radial extent at each contour is mapped to
a confocal prolate ellipsoid whose foci sit at the ends of the 22 mm active
freezing length.

In [9]:
import lung_chain

runs = [r for r in (lung_chain.chain_protocol(rep, 'VC3', -90.0)
                    for rep in ('N1', 'N2', 'N3')) if r is not None]

ALD50_C = float(tab_B.loc[tab_B.group == 'IceSphere triple', 'ALD50'].iloc[0])
ALD95_C = float(tab_B.loc[tab_B.group == 'IceSphere triple', 'ALD95'].iloc[0])

rows = []
for label, T in (('ice-ball edge (0 C)', 0.0), ('-20 C', -20.0), ('-40 C', -40.0),
                 ('ALD50', ALD50_C), ('ALD95', ALD95_C)):
    r_tip = np.mean([dr.find_isotherm_dist(r['T_min_field'], r['d_grid'], T)
                     for r in runs])
    g = dr.tip_to_ellipsoid(r_tip)
    rows.append(dict(contour=label, T_C=T, r_tip_mm=r_tip,
                     length_mm=g['length'], width_mm=g['width_max'],
                     volume_mm3=g['volume']))
geom = pd.DataFrame(rows)
print(geom.to_string(index=False, float_format=lambda v: f'{v:9.2f}'))

v = dict(zip(geom.contour, geom.volume_mm3))
print(f"\nALD95 volume / -40 C volume = {v['ALD95'] / v['-40 C']:.3f}")

            contour       T_C  r_tip_mm  length_mm  width_mm  volume_mm3
ice-ball edge (0 C)      0.00      5.00      27.56     16.61     3980.21
              -20 C    -20.00      2.83      25.01     11.89     1852.63
              -40 C    -40.00      1.53      23.58      8.49      889.50
              ALD50     -9.77      3.78      26.10     14.04     2695.11
              ALD95    -24.47      2.48      24.62     11.06     1576.17

ALD95 volume / -40 C volume = 1.772


## 5. Modified Stefan number

In [10]:
for medium, T_med, T_probe, label, T_fus, L_f in (
        ('PBS (Celsio)', 30.74, -79.0, 'CO2', -0.52, ct.L_WATER),
        ('PBS (IceSphere)', 27.27, -184.0, 'argon', -0.52, ct.L_WATER),
        ('GelMA (3D)', 29.23, -184.0, 'argon', -2.8, ct.L_GELMA)):
    st = dr.stefan_number(T_probe=T_probe, T_medium=T_med, T_fusion=T_fus,
                          c_p_solid=ct.C_ICE, c_p_liquid=ct.C_WATER, L_f=L_f)
    print(f'{medium:18s} {label:6s} T_M = {T_med:5.2f} C   St* = {st:.3f}')

PBS (Celsio)       CO2    T_M = 30.74 C   St* = 0.353
PBS (IceSphere)    argon  T_M = 27.27 C   St* = 0.852
GelMA (3D)         argon  T_M = 29.23 C   St* = 0.906
